In [ ]:
pip install tensorflow torch torchvision scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#VGG16
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50, Xception, InceptionV3, EfficientNetB0
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess
from tensorflow.keras.applications.xception import preprocess_input as xception_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, GlobalAveragePooling2D

# Set the dataset directory
dataset_dir = r'/content/drive/MyDrive/Colab Notebooks/cropped dataset'

# Image size for CNN models
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

def load_and_split_dataset(dataset_dir, test_size=0.2):
    datagen = ImageDataGenerator(preprocessing_function=None)
    data_generator = datagen.flow_from_directory(
        dataset_dir,
        target_size=IMG_SIZE,  # Image size as per the pre-trained model
        batch_size=1,  # Load one image at a time
        class_mode='categorical',
        shuffle=True)

    images, labels = [], []

    # Loop over the entire dataset and load images one by one
    for i in range(len(data_generator)):
        img, lbl = next(data_generator)  # Using next() function to get image and label
        images.append(img[0])  # Append the first (and only) image in the batch
        labels.append(lbl[0])  # Append the first (and only) label in the batch

    images = np.array(images)
    labels = np.array(labels)

    # Split dataset into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(images, labels, test_size=test_size, random_state=42)

    # Get class names
    class_names = list(data_generator.class_indices.keys())

    # Return training data, validation data, and class names
    return X_train, X_val, y_train, y_val, class_names

# Load and split dataset
X_train, X_val, y_train, y_val, class_names = load_and_split_dataset(dataset_dir)

# Ensure class_names is correctly returned and stored
print(class_names)  # This will print the class names to verify

# Function to build and compile a model using a pre-trained CNN
def build_model(base_model_fn, preprocess_fn, img_size=IMG_SIZE):
    # Load the pre-trained model
    base_model = base_model_fn(weights='imagenet', include_top=False, input_shape=img_size + (3,))

    # Freeze the pre-trained model layers
    base_model.trainable = False

    # Build the model
    model = Sequential([
        tf.keras.layers.InputLayer(input_shape=img_size + (3,)),
        tf.keras.layers.Lambda(preprocess_fn),  # Preprocessing specific to the model
        base_model,
        GlobalAveragePooling2D(),  # Use Global Average Pooling instead of Flatten for better generalization
        Dense(128, activation='relu'),
        Dense(y_train.shape[1], activation='softmax')  # Output layer for multi-class classification
    ])

    # Compile the model
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# Choose a pre-trained model (you can switch between these models)
model_name = 'VGG16'  # Options: 'VGG16', 'ResNet50', 'Xception', 'InceptionV3', 'EfficientNetB0'

if model_name == 'VGG16':
    model = build_model(VGG16, vgg16_preprocess)
elif model_name == 'ResNet50':
    model = build_model(ResNet50, resnet50_preprocess)
elif model_name == 'Xception':
    model = build_model(Xception, xception_preprocess)
elif model_name == 'InceptionV3':
    model = build_model(InceptionV3, inception_preprocess)
elif model_name == 'EfficientNetB0':
    model = build_model(EfficientNetB0, efficientnet_preprocess)

# Train the model
history = model.fit(X_train, y_train,
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    validation_data=(X_val, y_val))

# Evaluate the model
val_loss, val_acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {val_acc * 100:.2f}%")

Found 2199 images belonging to 2 classes.
['Abnormal', 'NORMAL']
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 49s 633ms/step - accuracy: 0.6844 - loss: 2.0707 - val_accuracy: 0.9409 - val_loss: 0.1642
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 46s 202ms/step - accuracy: 0.9525 - loss: 0.1275 - val_accuracy: 0.9705 - val_loss: 0.1113
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 21s 204ms/step - accuracy: 0.9737 - loss: 0.0695 - val_accuracy: 0.9727 - val_loss: 0.0813
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 11s 201ms/step - accuracy: 0.9775 - loss: 0.0710 - val_accuracy: 0.9341 - val_loss: 0.1415
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 11s 205ms/step - accuracy: 0.9718 - loss: 0.0800 - val_accuracy: 0.9841 - val_loss: 0.0606
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 11s 207ms/step - accuracy: 0.9855 - loss: 0.0445 - val_accuracy: 0.9795 - val_loss: 0.0626
Epoch 7/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 11s 207ms/step - accuracy: 0.9869 - loss: 0.0383 - val_accuracy: 0.9864 - val_loss: 0.0603
Epoch 8/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 20s 204ms/step - accuracy: 0.9774 - loss: 0.0567 - val_accu

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predict on the validation set
y_val_pred_prob = model.predict(X_val)

# Convert probabilities to class labels
y_val_pred = np.argmax(y_val_pred_prob, axis=1)
y_val_true = np.argmax(y_val, axis=1)

# Print classification report
print(classification_report(y_val_true, y_val_pred, target_names=class_names))

14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 233ms/step
              precision    recall  f1-score   support

    Abnormal       0.99      0.99      0.99       235
      NORMAL       0.99      0.99      0.99       205

    accuracy                           0.99       440
   macro avg       0.99      0.99      0.99       440
weighted avg       0.99      0.99      0.99       440



In [ ]:
#Resnet50
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50, Xception, InceptionV3, EfficientNetB0
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess
from tensorflow.keras.applications.xception import preprocess_input as xception_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, GlobalAveragePooling2D

# Set the dataset directory
dataset_dir = r'/content/drive/MyDrive/Colab Notebooks/cropped dataset'

# Image size for CNN models
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

# Load dataset manually and split it into train/validation sets
def load_and_split_dataset(dataset_dir, test_size=0.2):
    datagen = ImageDataGenerator(preprocessing_function=None)
    data_generator = datagen.flow_from_directory(
        dataset_dir,
        target_size=IMG_SIZE,  # Image size as per the pre-trained model
        batch_size=1,  # Load one image at a time
        class_mode='categorical',
        shuffle=True)

    images, labels = [], []

    # Loop over the entire dataset and load images one by one
    for i in range(len(data_generator)):
        img, lbl = next(data_generator)  # Using next() function to get image and label
        images.append(img[0])  # Append the first (and only) image in the batch
        labels.append(lbl[0])  # Append the first (and only) label in the batch

    images = np.array(images)
    labels = np.array(labels)

    # Split dataset into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(images, labels, test_size=test_size, random_state=42)

    # Get class names
    class_names = list(data_generator.class_indices.keys())

    # Return training data, validation data, and class names
    return X_train, X_val, y_train, y_val, class_names

# Load and split dataset
X_train, X_val, y_train, y_val, class_names = load_and_split_dataset(dataset_dir)

# Ensure class_names is correctly returned and stored
print(class_names)  # This will print the class names to verify

# Function to build and compile a model using a pre-trained CNN
def build_model(base_model_fn, preprocess_fn, img_size=IMG_SIZE):
    # Load the pre-trained model
    base_model = base_model_fn(weights='imagenet', include_top=False, input_shape=img_size + (3,))

    # Freeze the pre-trained model layers
    base_model.trainable = False

    # Build the model
    model = Sequential([
        tf.keras.layers.InputLayer(input_shape=img_size + (3,)),
        tf.keras.layers.Lambda(preprocess_fn),  # Preprocessing specific to the model
        base_model,
        GlobalAveragePooling2D(),  # Use Global Average Pooling instead of Flatten for better generalization
        Dense(128, activation='relu'),
        Dense(y_train.shape[1], activation='softmax')  # Output layer for multi-class classification
    ])

    # Compile the model
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# Choose a pre-trained model (you can switch between these models)
model_name = 'ResNet50'  # Options: 'VGG16', 'ResNet50', 'Xception', 'InceptionV3', 'EfficientNetB0'

if model_name == 'VGG16':
    model = build_model(VGG16, vgg16_preprocess)
elif model_name == 'ResNet50':
    model = build_model(ResNet50, resnet50_preprocess)
elif model_name == 'Xception':
    model = build_model(Xception, xception_preprocess)
elif model_name == 'InceptionV3':
    model = build_model(InceptionV3, inception_preprocess)
elif model_name == 'EfficientNetB0':
    model = build_model(EfficientNetB0, efficientnet_preprocess)

# Train the model
history = model.fit(X_train, y_train,
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    validation_data=(X_val, y_val))

# Evaluate the model
val_loss, val_acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {val_acc * 100:.2f}%")

Found 2199 images belonging to 2 classes.
['Abnormal', 'NORMAL']
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 32s 366ms/step - accuracy: 0.7722 - loss: 0.4740 - val_accuracy: 0.9682 - val_loss: 0.1171
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 6s 107ms/step - accuracy: 0.9559 - loss: 0.1040 - val_accuracy: 0.9432 - val_loss: 0.1474
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 106ms/step - accuracy: 0.9752 - loss: 0.0827 - val_accuracy: 0.9545 - val_loss: 0.1142
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 105ms/step - accuracy: 0.9773 - loss: 0.0646 - val_accuracy: 0.9818 - val_loss: 0.0686
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 109ms/step - accuracy: 0.9747 - loss: 0.0718 - val_accuracy: 0.9591 - val_loss: 0.1229
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 105ms/step - accuracy: 0.9776 - loss: 0.0658 - val_accuracy: 0.9750 - val_loss: 0.1162
Epoch 7/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 6s 107ms/step - accuracy: 0.9767 - loss: 0.0607 - val_accuracy: 0.9864 - val_loss: 0.0729
Epoch 8/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 6s 109ms/step - accuracy: 0.9898 - loss: 0.0369 - val_accurac

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predict on the validation set
y_val_pred_prob = model.predict(X_val)

# Convert probabilities to class labels
y_val_pred = np.argmax(y_val_pred_prob, axis=1)
y_val_true = np.argmax(y_val, axis=1)

# Print classification report
print(classification_report(y_val_true, y_val_pred, target_names=class_names))

14/14 ━━━━━━━━━━━━━━━━━━━━ 10s 463ms/step
              precision    recall  f1-score   support

    Abnormal       0.99      0.99      0.99       226
      NORMAL       0.99      0.99      0.99       214

    accuracy                           0.99       440
   macro avg       0.99      0.99      0.99       440
weighted avg       0.99      0.99      0.99       440



In [ ]:
#Xception
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50, Xception, InceptionV3, EfficientNetB0
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess
from tensorflow.keras.applications.xception import preprocess_input as xception_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, GlobalAveragePooling2D

# Set the dataset directory
dataset_dir = r'/content/drive/MyDrive/Colab Notebooks/cropped dataset'

# Image size for CNN models
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

# Load dataset manually and split it into train/validation sets
def load_and_split_dataset(dataset_dir, test_size=0.2):
    datagen = ImageDataGenerator(preprocessing_function=None)
    data_generator = datagen.flow_from_directory(
        dataset_dir,
        target_size=IMG_SIZE,  # Image size as per the pre-trained model
        batch_size=1,  # Load one image at a time
        class_mode='categorical',
        shuffle=True)

    images, labels = [], []

    # Loop over the entire dataset and load images one by one
    for i in range(len(data_generator)):
        img, lbl = next(data_generator)  # Using next() function to get image and label
        images.append(img[0])  # Append the first (and only) image in the batch
        labels.append(lbl[0])  # Append the first (and only) label in the batch

    images = np.array(images)
    labels = np.array(labels)

    # Split dataset into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(images, labels, test_size=test_size, random_state=42)

    # Get class names
    class_names = list(data_generator.class_indices.keys())

    # Return training data, validation data, and class names
    return X_train, X_val, y_train, y_val, class_names

# Load and split dataset
X_train, X_val, y_train, y_val, class_names = load_and_split_dataset(dataset_dir)

# Ensure class_names is correctly returned and stored
print(class_names)  # This will print the class names to verify

# Function to build and compile a model using a pre-trained CNN
def build_model(base_model_fn, preprocess_fn, img_size=IMG_SIZE):
    # Load the pre-trained model
    base_model = base_model_fn(weights='imagenet', include_top=False, input_shape=img_size + (3,))

    # Freeze the pre-trained model layers
    base_model.trainable = False

    # Build the model
    model = Sequential([
        tf.keras.layers.InputLayer(input_shape=img_size + (3,)),
        tf.keras.layers.Lambda(preprocess_fn),  # Preprocessing specific to the model
        base_model,
        GlobalAveragePooling2D(),  # Use Global Average Pooling instead of Flatten for better generalization
        Dense(128, activation='relu'),
        Dense(y_train.shape[1], activation='softmax')  # Output layer for multi-class classification
    ])

    # Compile the model
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# Choose a pre-trained model (you can switch between these models)
model_name = 'Xception'  # Options: 'VGG16', 'ResNet50', 'Xception', 'InceptionV3', 'EfficientNetB0'

if model_name == 'VGG16':
    model = build_model(VGG16, vgg16_preprocess)
elif model_name == 'ResNet50':
    model = build_model(ResNet50, resnet50_preprocess)
elif model_name == 'Xception':
    model = build_model(Xception, xception_preprocess)
elif model_name == 'InceptionV3':
    model = build_model(InceptionV3, inception_preprocess)
elif model_name == 'EfficientNetB0':
    model = build_model(EfficientNetB0, efficientnet_preprocess)

# Train the model
history = model.fit(X_train, y_train,
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    validation_data=(X_val, y_val))

# Evaluate the model
val_loss, val_acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {val_acc * 100:.2f}%")

Found 2199 images belonging to 2 classes.
['Abnormal', 'NORMAL']
83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 66s 833ms/step - accuracy: 0.7424 - loss: 0.5045 - val_accuracy: 0.9159 - val_loss: 0.2078
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 9s 159ms/step - accuracy: 0.9170 - loss: 0.2053 - val_accuracy: 0.9227 - val_loss: 0.1747
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 11s 175ms/step - accuracy: 0.9397 - loss: 0.1635 - val_accuracy: 0.9500 - val_loss: 0.1301
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 9s 160ms/step - accuracy: 0.9661 - loss: 0.1014 - val_accuracy: 0.9500 - val_loss: 0.1210
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 174ms/step - accuracy: 0.9659 - loss: 0.0937 - val_accuracy: 0.9386 - val_loss: 0.1530
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 172ms/step - accuracy: 0.9655 - loss: 0.0919 - val_accuracy: 0.9591 - val_loss: 0.1077
Epoch 7/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 9s 155ms/step - accuracy: 0.9842 - loss: 0.0560 - val_accuracy: 0.9591 - val_loss: 0.0846
Epoch 8/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 11s 171ms/step - accuracy: 0.9885 - loss: 0.0484 - val_accurac

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predict on the validation set
y_val_pred_prob = model.predict(X_val)

# Convert probabilities to class labels
y_val_pred = np.argmax(y_val_pred_prob, axis=1)
y_val_true = np.argmax(y_val, axis=1)

# Print classification report
print(classification_report(y_val_true, y_val_pred, target_names=class_names))

14/14 ━━━━━━━━━━━━━━━━━━━━ 12s 503ms/step
              precision    recall  f1-score   support

    Abnormal       1.00      0.92      0.96       249
      NORMAL       0.91      1.00      0.95       191

    accuracy                           0.95       440
   macro avg       0.95      0.96      0.95       440
weighted avg       0.96      0.95      0.95       440



In [ ]:
#EfficientNetB0
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50, Xception, InceptionV3, EfficientNetB0
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess
from tensorflow.keras.applications.xception import preprocess_input as xception_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, GlobalAveragePooling2D

# Set the dataset directory
dataset_dir = r'/content/drive/MyDrive/Colab Notebooks/cropped dataset'

# Image size for CNN models
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

# Load dataset manually and split it into train/validation sets
def load_and_split_dataset(dataset_dir, test_size=0.2):
    datagen = ImageDataGenerator(preprocessing_function=None)
    data_generator = datagen.flow_from_directory(
        dataset_dir,
        target_size=IMG_SIZE,  # Image size as per the pre-trained model
        batch_size=1,  # Load one image at a time
        class_mode='categorical',
        shuffle=True)

    images, labels = [], []

    # Loop over the entire dataset and load images one by one
    for i in range(len(data_generator)):
        img, lbl = next(data_generator)  # Using next() function to get image and label
        images.append(img[0])  # Append the first (and only) image in the batch
        labels.append(lbl[0])  # Append the first (and only) label in the batch

    images = np.array(images)
    labels = np.array(labels)

    # Split dataset into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(images, labels, test_size=test_size, random_state=42)

    # Get class names
    class_names = list(data_generator.class_indices.keys())

    # Return training data, validation data, and class names
    return X_train, X_val, y_train, y_val, class_names

# Load and split dataset
X_train, X_val, y_train, y_val, class_names = load_and_split_dataset(dataset_dir)

# Ensure class_names is correctly returned and stored
print(class_names)  # This will print the class names to verify

# Function to build and compile a model using a pre-trained CNN
def build_model(base_model_fn, preprocess_fn, img_size=IMG_SIZE):
    # Load the pre-trained model
    base_model = base_model_fn(weights='imagenet', include_top=False, input_shape=img_size + (3,))

    # Freeze the pre-trained model layers
    base_model.trainable = False

    # Build the model
    model = Sequential([
        tf.keras.layers.InputLayer(input_shape=img_size + (3,)),
        tf.keras.layers.Lambda(preprocess_fn),  # Preprocessing specific to the model
        base_model,
        GlobalAveragePooling2D(),  # Use Global Average Pooling instead of Flatten for better generalization
        Dense(128, activation='relu'),
        Dense(y_train.shape[1], activation='softmax')  # Output layer for multi-class classification
    ])

    # Compile the model
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# Choose a pre-trained model (you can switch between these models)
model_name = 'EfficientNetB0'  # Options: 'VGG16', 'ResNet50', 'Xception', 'InceptionV3', 'EfficientNetB0'

if model_name == 'VGG16':
    model = build_model(VGG16, vgg16_preprocess)
elif model_name == 'ResNet50':
    model = build_model(ResNet50, resnet50_preprocess)
elif model_name == 'Xception':
    model = build_model(Xception, xception_preprocess)
elif model_name == 'InceptionV3':
    model = build_model(InceptionV3, inception_preprocess)
elif model_name == 'EfficientNetB0':
    model = build_model(EfficientNetB0, efficientnet_preprocess)

# Train the model
history = model.fit(X_train, y_train,
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    validation_data=(X_val, y_val))

# Evaluate the model
val_loss, val_acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {val_acc * 100:.2f}%")

Found 2199 images belonging to 2 classes.
['Abnormal', 'NORMAL']
Epoch 1/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 46s 469ms/step - accuracy: 0.7875 - loss: 0.3936 - val_accuracy: 0.9795 - val_loss: 0.1003
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.9456 - loss: 0.1404 - val_accuracy: 0.9864 - val_loss: 0.0637
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - accuracy: 0.9547 - loss: 0.1295 - val_accuracy: 0.9795 - val_loss: 0.0563
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.9691 - loss: 0.0832 - val_accuracy: 0.9750 - val_loss: 0.0748
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.9600 - loss: 0.1081 - val_accuracy: 0.9295 - val_loss: 0.1440
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.9659 - loss: 0.1058 - val_accuracy: 0.9886 - val_loss: 0.0428
Epoch 7/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 48ms/step - accuracy: 0.9694 - loss: 0.0816 - val_accuracy: 0.9818 - val_loss: 0.0488
Epoch 8/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 51m

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predict on the validation set
y_val_pred_prob = model.predict(X_val)

# Convert probabilities to class labels
y_val_pred = np.argmax(y_val_pred_prob, axis=1)
y_val_true = np.argmax(y_val, axis=1)

# Print classification report
print(classification_report(y_val_true, y_val_pred, target_names=class_names))

14/14 ━━━━━━━━━━━━━━━━━━━━ 17s 519ms/step
              precision    recall  f1-score   support

    Abnormal       0.99      1.00      0.99       230
      NORMAL       1.00      0.99      0.99       210

    accuracy                           0.99       440
   macro avg       0.99      0.99      0.99       440
weighted avg       0.99      0.99      0.99       440



In [ ]:
pip install git+https://github.com/rcmalli/keras-squeezenet.git

  Cloning https://github.com/rcmalli/keras-squeezenet.git to /tmp/pip-req-build-8vwt4d2r
  Running command git clone --filter=blob:none --quiet https://github.com/rcmalli/keras-squeezenet.git /tmp/pip-req-build-8vwt4d2r
  Resolved https://github.com/rcmalli/keras-squeezenet.git to commit 4fb9cb7510ea0315303090edbc1bd97c2916af81
  Preparing metadata (setup.py) ... done
  Created wheel for keras_squeezenet: filename=keras_squeezenet-0.4-py3-none-any.whl size=4418 sha256=3291d9e4ab65eac38e5abb1980437efe26bc9a6f66462b321fbb4f019462325e
  Stored in directory: /tmp/pip-ephem-wheel-cache-j23uv2p5/wheels/7b/f5/12/4093c7cbe48ef117383eddc06c4fd74a21c2cadbf5f42be163
Successfully built keras_squeezenet


In [ ]:
pip install keras-squeezenet

In [ ]:
#Alexnet
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.models import Sequential

# Set the dataset directory
dataset_dir = r'/content/drive/MyDrive/Colab Notebooks/cropped dataset'

# Image size for CNN models
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

# Load dataset manually and split it into train/validation sets
def load_and_split_dataset(dataset_dir, test_size=0.2):
    datagen = ImageDataGenerator(preprocessing_function=None)
    data_generator = datagen.flow_from_directory(
        dataset_dir,
        target_size=IMG_SIZE,  # Image size as per the pre-trained model
        batch_size=1,  # Load one image at a time
        class_mode='categorical',
        shuffle=True)

    images, labels = [], []

    # Loop over the entire dataset and load images one by one
    for i in range(len(data_generator)):
        img, lbl = next(data_generator)  # Using next() function to get image and label
        images.append(img[0])  # Append the first (and only) image in the batch
        labels.append(lbl[0])  # Append the first (and only) label in the batch

    images = np.array(images)
    labels = np.array(labels)

    # Split dataset into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(images, labels, test_size=test_size, random_state=42)

    # Get class names
    class_names = list(data_generator.class_indices.keys())

    # Return training data, validation data, and class names
    return X_train, X_val, y_train, y_val, class_names

# Load and split dataset
X_train, X_val, y_train, y_val, class_names = load_and_split_dataset(dataset_dir)

# Ensure class_names is correctly returned and stored
print(class_names)  # This will print the class names to verify

# Function to build AlexNet model
def build_alexnet(input_shape, num_classes):
    model = Sequential([
        Conv2D(96, kernel_size=(11, 11), strides=4, activation='relu', input_shape=input_shape),
        MaxPooling2D(pool_size=(3, 3), strides=2),

        Conv2D(256, kernel_size=(5, 5), padding='same', activation='relu'),
        MaxPooling2D(pool_size=(3, 3), strides=2),

        Conv2D(384, kernel_size=(3, 3), padding='same', activation='relu'),
        Conv2D(384, kernel_size=(3, 3), padding='same', activation='relu'),
        Conv2D(256, kernel_size=(3, 3), padding='same', activation='relu'),
        MaxPooling2D(pool_size=(3, 3), strides=2),

        Flatten(),
        Dense(4096, activation='relu'),
        Dropout(0.5),
        Dense(4096, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Function to build a simple SqueezeNet-like model
def build_squeezenet(input_shape, num_classes):
    model = Sequential([
        Conv2D(64, kernel_size=(3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D(pool_size=(3, 3)),

        Conv2D(128, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(3, 3)),

        Conv2D(256, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(3, 3)),

        Flatten(),
        Dense(512, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Choose a model (you can switch between these models)
model_name = 'AlexNet'  # Options: 'AlexNet', 'SqueezeNet', 'InceptionV3' (for GoogLeNet approximation)

if model_name == 'AlexNet':
    model = build_alexnet(IMG_SIZE + (3,), y_train.shape[1])
elif model_name == 'SqueezeNet':
    model = build_squeezenet(IMG_SIZE + (3,), y_train.shape[1])
elif model_name == 'InceptionV3':  # Approximation for GoogLeNet
    from tensorflow.keras.applications import InceptionV3
    from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
    model = Sequential([
        tf.keras.layers.InputLayer(input_shape=IMG_SIZE + (3,)),
        tf.keras.layers.Lambda(inception_preprocess),
        InceptionV3(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,)),
        GlobalAveragePooling2D(),
        Dense(128, activation='relu'),
        Dense(y_train.shape[1], activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train,
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    validation_data=(X_val, y_val))

# Evaluate the model
val_loss, val_acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {val_acc * 100:.2f}%")

Found 2199 images belonging to 2 classes.
['Abnormal', 'NORMAL']


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 25s 270ms/step - accuracy: 0.5178 - loss: 112.5483 - val_accuracy: 0.6182 - val_loss: 0.6156
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 22s 59ms/step - accuracy: 0.6804 - loss: 0.5773 - val_accuracy: 0.6273 - val_loss: 0.5543
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.6872 - loss: 0.5504 - val_accuracy: 0.6841 - val_loss: 0.5419
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - accuracy: 0.7185 - loss: 0.5116 - val_accuracy: 0.5568 - val_loss: 0.6399
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - accuracy: 0.6643 - loss: 0.5651 - val_accuracy: 0.6273 - val_loss: 0.5596
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.5592 - loss: 6.3890 - val_accuracy: 0.5727 - val_loss: 0.6905
Epoch 7/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - accuracy: 0.5319 - loss: 0.7625 - val_accuracy: 0.5727 - val_loss: 0.7001
Epoch 8/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.4837 - loss: 0.6941 - val_accuracy: 0.572

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predict on the validation set
y_val_pred_prob = model.predict(X_val)

# Convert probabilities to class labels
y_val_pred = np.argmax(y_val_pred_prob, axis=1)
y_val_true = np.argmax(y_val, axis=1)

# Print classification report
print(classification_report(y_val_true, y_val_pred, target_names=class_names))

14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 89ms/step
              precision    recall  f1-score   support

    Abnormal       0.57      1.00      0.73       253
      NORMAL       0.00      0.00      0.00       187

    accuracy                           0.57       440
   macro avg       0.29      0.50      0.36       440
weighted avg       0.33      0.57      0.42       440



In [ ]:
#Googlenet
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.models import Sequential

# Set the dataset directory
dataset_dir = r'/content/drive/MyDrive/Colab Notebooks/cropped dataset'

# Image size for CNN models
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

# Load dataset manually and split it into train/validation sets
def load_and_split_dataset(dataset_dir, test_size=0.2):
    datagen = ImageDataGenerator(preprocessing_function=None)
    data_generator = datagen.flow_from_directory(
        dataset_dir,
        target_size=IMG_SIZE,  # Image size as per the pre-trained model
        batch_size=1,  # Load one image at a time
        class_mode='categorical',
        shuffle=True)

    images, labels = [], []

    # Loop over the entire dataset and load images one by one
    for i in range(len(data_generator)):
        img, lbl = next(data_generator)  # Using next() function to get image and label
        images.append(img[0])  # Append the first (and only) image in the batch
        labels.append(lbl[0])  # Append the first (and only) label in the batch

    images = np.array(images)
    labels = np.array(labels)

    # Split dataset into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(images, labels, test_size=test_size, random_state=42)

    # Get class names
    class_names = list(data_generator.class_indices.keys())

    # Return training data, validation data, and class names
    return X_train, X_val, y_train, y_val, class_names

# Load and split dataset
X_train, X_val, y_train, y_val, class_names = load_and_split_dataset(dataset_dir)

# Ensure class_names is correctly returned and stored
print(class_names)  # This will print the class names to verify

# Function to build AlexNet model
def build_alexnet(input_shape, num_classes):
    model = Sequential([
        Conv2D(96, kernel_size=(11, 11), strides=4, activation='relu', input_shape=input_shape),
        MaxPooling2D(pool_size=(3, 3), strides=2),

        Conv2D(256, kernel_size=(5, 5), padding='same', activation='relu'),
        MaxPooling2D(pool_size=(3, 3), strides=2),

        Conv2D(384, kernel_size=(3, 3), padding='same', activation='relu'),
        Conv2D(384, kernel_size=(3, 3), padding='same', activation='relu'),
        Conv2D(256, kernel_size=(3, 3), padding='same', activation='relu'),
        MaxPooling2D(pool_size=(3, 3), strides=2),

        Flatten(),
        Dense(4096, activation='relu'),
        Dropout(0.5),
        Dense(4096, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Function to build a simple SqueezeNet-like model
def build_squeezenet(input_shape, num_classes):
    model = Sequential([
        Conv2D(64, kernel_size=(3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D(pool_size=(3, 3)),

        Conv2D(128, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(3, 3)),

        Conv2D(256, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(3, 3)),

        Flatten(),
        Dense(512, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Choose a model (you can switch between these models)
model_name = 'InceptionV3'  # Options: 'AlexNet', 'SqueezeNet', 'InceptionV3' (for GoogLeNet approximation)

if model_name == 'AlexNet':
    model = build_alexnet(IMG_SIZE + (3,), y_train.shape[1])
elif model_name == 'SqueezeNet':
    model = build_squeezenet(IMG_SIZE + (3,), y_train.shape[1])
elif model_name == 'InceptionV3':  # Approximation for GoogLeNet
    from tensorflow.keras.applications import InceptionV3
    from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
    model = Sequential([
        tf.keras.layers.InputLayer(input_shape=IMG_SIZE + (3,)),
        tf.keras.layers.Lambda(inception_preprocess),
        InceptionV3(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,)),
        GlobalAveragePooling2D(),
        Dense(128, activation='relu'),
        Dense(y_train.shape[1], activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train,
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    validation_data=(X_val, y_val))

# Evaluate the model
val_loss, val_acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {val_acc * 100:.2f}%")

Found 2199 images belonging to 2 classes.
['Abnormal', 'NORMAL']


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 162s 2s/step - accuracy: 0.8457 - loss: 0.3452 - val_accuracy: 0.3523 - val_loss: 11.9721
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 30s 248ms/step - accuracy: 0.9817 - loss: 0.0680 - val_accuracy: 0.8705 - val_loss: 0.5832
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 14s 252ms/step - accuracy: 0.9912 - loss: 0.0321 - val_accuracy: 0.7045 - val_loss: 2.0624
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 20s 246ms/step - accuracy: 0.9859 - loss: 0.0514 - val_accuracy: 0.7977 - val_loss: 1.4640
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 21s 253ms/step - accuracy: 0.9931 - loss: 0.0200 - val_accuracy: 0.9750 - val_loss: 0.0698
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 14s 249ms/step - accuracy: 0.9984 - loss: 0.0043 - val_accuracy: 0.8909 - val_loss: 0.5399
Epoch 7/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 20s 247ms/step - accuracy: 0.9647 - loss: 0.1196 - val_accuracy: 0.6000 - val_loss: 11.4787
Epoch 8/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 21s 254m

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predict on the validation set
y_val_pred_prob = model.predict(X_val)

# Convert probabilities to class labels
y_val_pred = np.argmax(y_val_pred_prob, axis=1)
y_val_true = np.argmax(y_val, axis=1)

# Print classification report
print(classification_report(y_val_true, y_val_pred, target_names=class_names))

14/14 ━━━━━━━━━━━━━━━━━━━━ 14s 520ms/step
              precision    recall  f1-score   support

    Abnormal       1.00      1.00      1.00       227
      NORMAL       1.00      1.00      1.00       213

    accuracy                           1.00       440
   macro avg       1.00      1.00      1.00       440
weighted avg       1.00      1.00      1.00       440



In [ ]:
#Squeezenet
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.models import Sequential

# Set the dataset directory
dataset_dir = r'/content/drive/MyDrive/Colab Notebooks/cropped dataset'

# Image size for CNN models
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

# Load dataset manually and split it into train/validation sets
def load_and_split_dataset(dataset_dir, test_size=0.2):
    datagen = ImageDataGenerator(preprocessing_function=None)
    data_generator = datagen.flow_from_directory(
        dataset_dir,
        target_size=IMG_SIZE,  # Image size as per the pre-trained model
        batch_size=1,  # Load one image at a time
        class_mode='categorical',
        shuffle=True)

    images, labels = [], []

    # Loop over the entire dataset and load images one by one
    for i in range(len(data_generator)):
        img, lbl = next(data_generator)  # Using next() function to get image and label
        images.append(img[0])  # Append the first (and only) image in the batch
        labels.append(lbl[0])  # Append the first (and only) label in the batch

    images = np.array(images)
    labels = np.array(labels)

    # Split dataset into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(images, labels, test_size=test_size, random_state=42)

    # Get class names
    class_names = list(data_generator.class_indices.keys())

    # Return training data, validation data, and class names
    return X_train, X_val, y_train, y_val, class_names

# Load and split dataset
X_train, X_val, y_train, y_val, class_names = load_and_split_dataset(dataset_dir)

# Ensure class_names is correctly returned and stored
print(class_names)  # This will print the class names to verify

# Function to build AlexNet model
def build_alexnet(input_shape, num_classes):
    model = Sequential([
        Conv2D(96, kernel_size=(11, 11), strides=4, activation='relu', input_shape=input_shape),
        MaxPooling2D(pool_size=(3, 3), strides=2),

        Conv2D(256, kernel_size=(5, 5), padding='same', activation='relu'),
        MaxPooling2D(pool_size=(3, 3), strides=2),

        Conv2D(384, kernel_size=(3, 3), padding='same', activation='relu'),
        Conv2D(384, kernel_size=(3, 3), padding='same', activation='relu'),
        Conv2D(256, kernel_size=(3, 3), padding='same', activation='relu'),
        MaxPooling2D(pool_size=(3, 3), strides=2),

        Flatten(),
        Dense(4096, activation='relu'),
        Dropout(0.5),
        Dense(4096, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Function to build a simple SqueezeNet-like model
def build_squeezenet(input_shape, num_classes):
    model = Sequential([
        Conv2D(64, kernel_size=(3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D(pool_size=(3, 3)),

        Conv2D(128, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(3, 3)),

        Conv2D(256, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(3, 3)),

        Flatten(),
        Dense(512, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Choose a model (you can switch between these models)
model_name = 'SqueezeNet'  # Options: 'AlexNet', 'SqueezeNet', 'InceptionV3' (for GoogLeNet approximation)

if model_name == 'AlexNet':
    model = build_alexnet(IMG_SIZE + (3,), y_train.shape[1])
elif model_name == 'SqueezeNet':
    model = build_squeezenet(IMG_SIZE + (3,), y_train.shape[1])
elif model_name == 'InceptionV3':  # Approximation for GoogLeNet
    from tensorflow.keras.applications import InceptionV3
    from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
    model = Sequential([
        tf.keras.layers.InputLayer(input_shape=IMG_SIZE + (3,)),
        tf.keras.layers.Lambda(inception_preprocess),
        InceptionV3(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,)),
        GlobalAveragePooling2D(),
        Dense(128, activation='relu'),
        Dense(y_train.shape[1], activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train,
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    validation_data=(X_val, y_val))

# Evaluate the model
val_loss, val_acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {val_acc * 100:.2f}%")

Found 2199 images belonging to 2 classes.
['Abnormal', 'NORMAL']


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 22s 253ms/step - accuracy: 0.6292 - loss: 99.2617 - val_accuracy: 0.7841 - val_loss: 0.3707
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 65ms/step - accuracy: 0.8320 - loss: 0.3334 - val_accuracy: 0.8477 - val_loss: 0.3046
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 64ms/step - accuracy: 0.8858 - loss: 0.2356 - val_accuracy: 0.8932 - val_loss: 0.2109
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 66ms/step - accuracy: 0.9358 - loss: 0.1689 - val_accuracy: 0.9841 - val_loss: 0.0963
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 65ms/step - accuracy: 0.9760 - loss: 0.0813 - val_accuracy: 0.9727 - val_loss: 0.1214
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 66ms/step - accuracy: 0.9886 - loss: 0.0421 - val_accuracy: 0.9841 - val_loss: 0.0547
Epoch 7/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 66ms/step - accuracy: 0.9892 - loss: 0.0335 - val_accuracy: 0.9864 - val_loss: 0.0578
Epoch 8/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 65ms/step - accuracy: 0.9959 - loss: 0.0203 - val_accuracy: 0.9773 

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predict on the validation set
y_val_pred_prob = model.predict(X_val)

# Convert probabilities to class labels
y_val_pred = np.argmax(y_val_pred_prob, axis=1)
y_val_true = np.argmax(y_val, axis=1)

# Print classification report
print(classification_report(y_val_true, y_val_pred, target_names=class_names))

14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 74ms/step
              precision    recall  f1-score   support

    Abnormal       1.00      0.94      0.97       219
      NORMAL       0.94      1.00      0.97       221

    accuracy                           0.97       440
   macro avg       0.97      0.97      0.97       440
weighted avg       0.97      0.97      0.97       440



In [ ]:
#VGG16+LSTM
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Bidirectional, GlobalAveragePooling2D, TimeDistributed, Flatten
from tensorflow.keras.layers import Reshape

# Set the dataset directory
dataset_dir = r'/content/drive/MyDrive/Colab Notebooks/cropped dataset'

# Image size for CNN models
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

# Load dataset manually and split it into train/validation sets
def load_and_split_dataset(dataset_dir, test_size=0.2):
    datagen = ImageDataGenerator(preprocessing_function=None)
    data_generator = datagen.flow_from_directory(
        dataset_dir,
        target_size=IMG_SIZE,  # Image size as per the pre-trained model
        batch_size=1,  # Load one image at a time
        class_mode='categorical',
        shuffle=True)

    images, labels = [], []

    # Loop over the entire dataset and load images one by one
    for i in range(len(data_generator)):
        img, lbl = next(data_generator)  # Using next() function to get image and label
        images.append(img[0])  # Append the first (and only) image in the batch
        labels.append(lbl[0])  # Append the first (and only) label in the batch

    images = np.array(images)
    labels = np.array(labels)

    # Split dataset into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(images, labels, test_size=test_size, random_state=42)

    # Get class names
    class_names = list(data_generator.class_indices.keys())

    # Return training data, validation data, and class names
    return X_train, X_val, y_train, y_val, class_names

# Load and split dataset
X_train, X_val, y_train, y_val, class_names = load_and_split_dataset(dataset_dir)

# Ensure class_names is correctly returned and stored
print(class_names)  # This will print the class names to verify

# Function to build and compile a model with LSTM or BiLSTM
def build_model_lstm(base_model_fn, preprocess_fn, lstm_type='LSTM', img_size=IMG_SIZE, timesteps=1):
    # Load the pre-trained model (e.g., VGG16)
    base_model = base_model_fn(weights='imagenet', include_top=False, input_shape=img_size + (3,))

    # Freeze the pre-trained model layers
    base_model.trainable = False

    # Build the model
    model = Sequential([
        tf.keras.layers.InputLayer(input_shape=img_size + (3,)),
        tf.keras.layers.Lambda(preprocess_fn),  # Preprocessing specific to the model
        base_model,
        GlobalAveragePooling2D(),  # Extract features from the CNN
        Reshape((timesteps, -1)),  # Reshape to (timesteps, features) for LSTM input

        # LSTM or BiLSTM layer based on the choice
        LSTM(128, return_sequences=False) if lstm_type == 'LSTM' else Bidirectional(LSTM(128, return_sequences=False)),

        Dense(128, activation='relu'),
        Dense(y_train.shape[1], activation='softmax')  # Output layer for multi-class classification
    ])

    # Compile the model
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# Choose a pre-trained model (using VGG16 for feature extraction)
model_name = 'VGG16'

# Choose LSTM or BiLSTM
lstm_type = 'LSTM'  # Options: 'LSTM', 'BiLSTM'

if model_name == 'VGG16':
    if lstm_type == 'LSTM':
        model = build_model_lstm(VGG16, vgg16_preprocess, lstm_type='LSTM')
    elif lstm_type == 'BiLSTM':
        model = build_model_lstm(VGG16, vgg16_preprocess, lstm_type='BiLSTM')

# Train the model
history = model.fit(X_train, y_train,
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    validation_data=(X_val, y_val))

# Evaluate the model
val_loss, val_acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {val_acc * 100:.2f}%")

Found 2199 images belonging to 2 classes.
['Abnormal', 'NORMAL']


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 12s 177ms/step - accuracy: 0.8739 - loss: 0.3227 - val_accuracy: 0.9591 - val_loss: 0.0892
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 174ms/step - accuracy: 0.9770 - loss: 0.0652 - val_accuracy: 0.9591 - val_loss: 0.0920
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 9s 157ms/step - accuracy: 0.9822 - loss: 0.0501 - val_accuracy: 0.9773 - val_loss: 0.0598
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 11s 173ms/step - accuracy: 0.9845 - loss: 0.0504 - val_accuracy: 0.9682 - val_loss: 0.0861
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 9s 161ms/step - accuracy: 0.9862 - loss: 0.0368 - val_accuracy: 0.9841 - val_loss: 0.0484
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 162ms/step - accuracy: 0.9884 - loss: 0.0344 - val_accuracy: 0.9705 - val_loss: 0.0689
Epoch 7/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 9s 162ms/step - accuracy: 0.9916 - loss: 0.0301 - val_accuracy: 0.9682 - val_loss: 0.0544
Epoch 8/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 9s 160ms/step - accuracy: 0.9892 - loss: 0.0295 - val_accuracy

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predict on the validation set
y_val_pred_prob = model.predict(X_val)

# Convert probabilities to class labels
y_val_pred = np.argmax(y_val_pred_prob, axis=1)
y_val_true = np.argmax(y_val, axis=1)

# Print classification report
print(classification_report(y_val_true, y_val_pred, target_names=class_names))

14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 151ms/step
              precision    recall  f1-score   support

    Abnormal       0.94      1.00      0.97       238
      NORMAL       0.99      0.93      0.96       202

    accuracy                           0.96       440
   macro avg       0.97      0.96      0.96       440
weighted avg       0.97      0.96      0.96       440



In [ ]:
#VGG16+Bilstm
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Bidirectional, GlobalAveragePooling2D, TimeDistributed, Flatten
from tensorflow.keras.layers import Reshape

# Set the dataset directory
dataset_dir = r'/content/drive/MyDrive/Colab Notebooks/cropped dataset'

# Image size for CNN models
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

# Load dataset manually and split it into train/validation sets
def load_and_split_dataset(dataset_dir, test_size=0.2):
    datagen = ImageDataGenerator(preprocessing_function=None)
    data_generator = datagen.flow_from_directory(
        dataset_dir,
        target_size=IMG_SIZE,  # Image size as per the pre-trained model
        batch_size=1,  # Load one image at a time
        class_mode='categorical',
        shuffle=True)

    images, labels = [], []

    # Loop over the entire dataset and load images one by one
    for i in range(len(data_generator)):
        img, lbl = next(data_generator)  # Using next() function to get image and label
        images.append(img[0])  # Append the first (and only) image in the batch
        labels.append(lbl[0])  # Append the first (and only) label in the batch

    images = np.array(images)
    labels = np.array(labels)

    # Split dataset into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(images, labels, test_size=test_size, random_state=42)

    # Get class names
    class_names = list(data_generator.class_indices.keys())

    # Return training data, validation data, and class names
    return X_train, X_val, y_train, y_val, class_names

# Load and split dataset
X_train, X_val, y_train, y_val, class_names = load_and_split_dataset(dataset_dir)

# Ensure class_names is correctly returned and stored
print(class_names)  # This will print the class names to verify

# Function to build and compile a model with LSTM or BiLSTM
def build_model_lstm(base_model_fn, preprocess_fn, lstm_type='LSTM', img_size=IMG_SIZE, timesteps=1):
    # Load the pre-trained model (e.g., VGG16)
    base_model = base_model_fn(weights='imagenet', include_top=False, input_shape=img_size + (3,))

    # Freeze the pre-trained model layers
    base_model.trainable = False

    # Build the model
    model = Sequential([
        tf.keras.layers.InputLayer(input_shape=img_size + (3,)),
        tf.keras.layers.Lambda(preprocess_fn),  # Preprocessing specific to the model
        base_model,
        GlobalAveragePooling2D(),  # Extract features from the CNN
        Reshape((timesteps, -1)),  # Reshape to (timesteps, features) for LSTM input

        # LSTM or BiLSTM layer based on the choice
        LSTM(128, return_sequences=False) if lstm_type == 'LSTM' else Bidirectional(LSTM(128, return_sequences=False)),

        Dense(128, activation='relu'),
        Dense(y_train.shape[1], activation='softmax')  # Output layer for multi-class classification
    ])

    # Compile the model
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# Choose a pre-trained model (using VGG16 for feature extraction)
model_name = 'VGG16'

# Choose LSTM or BiLSTM
lstm_type = 'BiLSTM'  # Options: 'LSTM', 'BiLSTM'

if model_name == 'VGG16':
    if lstm_type == 'LSTM':
        model = build_model_lstm(VGG16, vgg16_preprocess, lstm_type='LSTM')
    elif lstm_type == 'BiLSTM':
        model = build_model_lstm(VGG16, vgg16_preprocess, lstm_type='BiLSTM')

# Train the model
history = model.fit(X_train, y_train,
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    validation_data=(X_val, y_val))

# Evaluate the model
val_loss, val_acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {val_acc * 100:.2f}%")

Found 2199 images belonging to 2 classes.
['Abnormal', 'NORMAL']


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 24s 292ms/step - accuracy: 0.7899 - loss: 0.4005 - val_accuracy: 0.9386 - val_loss: 0.1565
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 8s 154ms/step - accuracy: 0.9666 - loss: 0.0916 - val_accuracy: 0.9795 - val_loss: 0.0571
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 157ms/step - accuracy: 0.9787 - loss: 0.0615 - val_accuracy: 0.9795 - val_loss: 0.0475
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 158ms/step - accuracy: 0.9817 - loss: 0.0434 - val_accuracy: 0.9795 - val_loss: 0.0506
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 11s 176ms/step - accuracy: 0.9867 - loss: 0.0382 - val_accuracy: 0.9818 - val_loss: 0.0390
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 9s 158ms/step - accuracy: 0.9919 - loss: 0.0265 - val_accuracy: 0.9909 - val_loss: 0.0387
Epoch 7/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 11s 175ms/step - accuracy: 0.9880 - loss: 0.0286 - val_accuracy: 0.9932 - val_loss: 0.0266
Epoch 8/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 9s 160ms/step - accuracy: 0.9897 - loss: 0.0293 - val_accurac

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predict on the validation set
y_val_pred_prob = model.predict(X_val)

# Convert probabilities to class labels
y_val_pred = np.argmax(y_val_pred_prob, axis=1)
y_val_true = np.argmax(y_val, axis=1)

# Print classification report
print(classification_report(y_val_true, y_val_pred, target_names=class_names))

14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 143ms/step
              precision    recall  f1-score   support

    Abnormal       1.00      0.99      0.99       226
      NORMAL       0.99      1.00      0.99       214

    accuracy                           0.99       440
   macro avg       0.99      0.99      0.99       440
weighted avg       0.99      0.99      0.99       440

